In [4]:
import mysql.connector

# 连接 MySQL 数据库
conn = mysql.connector.connect(
    host="localhost",
    user="root",
    password="20041025",  # 替换为你自己的密码
    database="ucb_db"
)

cursor = conn.cursor()

# 查询所有描述
cursor.execute("SELECT description FROM business_descriptions")
descriptions = [row[0] for row in cursor.fetchall() if row[0]]

# 筛选包含 Los Angeles 的描述
la_descriptions = [desc for desc in descriptions if "Los Angeles" in desc]

print(f"✅ 共找到 {len(la_descriptions)} 条包含 'Los Angeles' 的描述")
for desc in la_descriptions[:5]:
    print("📝", desc)


✅ 共找到 7 条包含 'Los Angeles' 的描述
📝  This textile exporter located in Los Angeles, CA 90023 specializes in providing high-quality fabrics and materials to global markets.
📝  This vibrant Korean restaurant in Los Angeles, CA 90005, offers an authentic taste of Korea with a diverse menu featuring traditional dishes and modern interpretations, perfect for both casual dining and special occasions.
📝  This fabric store in Los Angeles, CA 90021 offers a wide selection of high-quality textiles, perfect for all your sewing and crafting needs.
📝  This fabric store in Los Angeles, CA 90021 offers a wide selection of high-quality textiles and materials for all your sewing and crafting needs.
📝  Discover a vibrant bead store in Los Angeles, CA 90014, offering a wide selection of high-quality beads, supplies, and crafting tools for all your creative projects.


In [5]:
import mysql.connector
import openai
import time
import json
from tqdm import tqdm
from openai import AzureOpenAI

# ====== Azure OpenAI 配置 ======
client = AzureOpenAI(
    api_key="609ced4d971240b8a08f7fb0e6d846ea",
    api_version="2024-08-01-preview",
    azure_endpoint="https://promptdelta-nc.openai.azure.com",  # 不要加 /v1
)

deployment_name = "gpt-4o-mini"  # 在 Azure Portal 的 Deployments 中查看


# 连接 MySQL 数据库
conn = mysql.connector.connect(
    host="localhost",
    user="root",
    password="20041025",
    database="ucb_db"
)
cursor = conn.cursor()

# 获取所有字段（包括 description）
cursor.execute("SELECT * FROM business_descriptions")
column_names = [desc[0] for desc in cursor.description]
rows = cursor.fetchall()

# 定义 GPT 判断函数
def is_los_angeles(description, client, deployment_name):
    # 构造 prompt
    prompt = (
        "Does the following business description mention that it is located in Los Angeles?\n"
        "Just answer 'yes' or 'no'.\n\n"
        f"Description: {description}"
    )

    try:
        response = client.chat.completions.create(
            model=deployment_name,
            messages=[
                {"role": "system", "content": "You are a helpful assistant that extracts geographic location from business descriptions."},
                {"role": "user", "content": prompt}
            ],
            temperature=0.0,
            max_tokens=5
        )
        return response.choices[0].message.content.strip().lower().startswith("yes")

    except Exception as e:
        print(f"❌ Error determining location from description: {e}")
        return False

# ✅ 用于保存符合条件的记录
la_data = []

# ✅ 遍历记录进行 GPT 判断
for i, row in enumerate(rows[:100]):  # 可根据需要扩大范围
    record = dict(zip(column_names, row))
    description = record.get("description", "")

    if description and is_los_angeles(description, client, deployment_name):
        la_data.append(record)
        print(f"✅ [{i}] 是洛杉矶: {description}")
    else:
        print(f"❌ [{i}] 不是洛杉矶")

print(f"\n🎯 最终保留 {len(la_data)} 条位于洛杉矶的完整记录")


✅ [0] 是洛杉矶:  This textile exporter located in Los Angeles, CA 90023 specializes in providing high-quality fabrics and materials to global markets.
✅ [1] 是洛杉矶:  This vibrant Korean restaurant in Los Angeles, CA 90005, offers an authentic taste of Korea with a diverse menu featuring traditional dishes and modern interpretations, perfect for both casual dining and special occasions.
✅ [2] 是洛杉矶:  This fabric store in Los Angeles, CA 90021 offers a wide selection of high-quality textiles, perfect for all your sewing and crafting needs.
✅ [3] 是洛杉矶:  This fabric store in Los Angeles, CA 90021 offers a wide selection of high-quality textiles and materials for all your sewing and crafting needs.
❌ [4] 不是洛杉矶
❌ [5] 不是洛杉矶
❌ [6] 不是洛杉矶
❌ [7] 不是洛杉矶
❌ [8] 不是洛杉矶
❌ [9] 不是洛杉矶
❌ [10] 不是洛杉矶
❌ [11] 不是洛杉矶
❌ [12] 不是洛杉矶
❌ [13] 不是洛杉矶
❌ [14] 不是洛杉矶
❌ [15] 不是洛杉矶
❌ [16] 不是洛杉矶
❌ [17] 不是洛杉矶
❌ [18] 不是洛杉矶
❌ [19] 不是洛杉矶
❌ [20] 不是洛杉矶
❌ [21] 不是洛杉矶
❌ [22] 不是洛杉矶
❌ [23] 不是洛杉矶
❌ [24] 不是洛杉矶
❌ [25] 不是洛杉矶
❌ [26] 不是洛杉矶
❌ [27] 不是洛杉

In [6]:
la_data

[{'name': 'City Textile',
  'gmap_id': 'gmap_0',
  'description': ' This textile exporter located in Los Angeles, CA 90023 specializes in providing high-quality fabrics and materials to global markets.',
  'num_of_reviews': 6,
  'hours': 'null',
  'MISC': 'null',
  'state': 'Open now'},
 {'name': 'San Soo Dang',
  'gmap_id': 'gmap_1',
  'description': ' This vibrant Korean restaurant in Los Angeles, CA 90005, offers an authentic taste of Korea with a diverse menu featuring traditional dishes and modern interpretations, perfect for both casual dining and special occasions.',
  'num_of_reviews': 18,
  'hours': '[["Thursday", "6:30AM–6PM"], ["Friday", "6:30AM–6PM"], ["Saturday", "6:30AM–6PM"], ["Sunday", "7AM–12PM"], ["Monday", "Closed"], ["Tuesday", "6:30AM–6PM"], ["Wednesday", "6:30AM–6PM"]]',
  'MISC': '{"Amenities": ["Good for kids"], "Offerings": ["Comfort food"], "Atmosphere": ["Casual"], "Accessibility": ["Wheelchair accessible entrance"], "Service options": ["Takeout", "Dine-in", 

In [7]:
import sqlite3

# 尝试连接 SQLite 数据库
db_path = "../query_dataset/review_query.db"
conn = sqlite3.connect(db_path)
cursor = conn.cursor()

# 查看数据库中所有表
cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
tables = cursor.fetchall()
print("📋 数据库中包含的表：", tables)


📋 数据库中包含的表： [('review',)]


In [8]:
cursor.execute("SELECT * FROM review LIMIT 5;")
rows = cursor.fetchall()
colnames = [desc[0] for desc in cursor.description]

print("字段名：", colnames)
for row in rows:
    print(row)


字段名： ['name', 'time', 'rating', 'text', 'gmap_id']
('Michael Rizal', 'September 03, 2020 at 04:15 PM', 5, 'Located in the vibrant area of Los Angeles, CA 90023, this company truly stands out. "Great company. Amazing customer service and they always have what we need in stock. Sometimes, we’d ask to hold for future orders and they will! Miss Jane is very helpful and great communicator."', 'gmap_0')
('Faranak Rafizadeh', '2021-04-12 17:07:52', 5, 'Los Angeles is known for its vibrant culture and friendly atmosphere. "Nice people helpful."', 'gmap_0')
('Javier Perez', '2018-04-23 16:24:26', 5, 'I had a fantastic experience at this amazing spot in Los Angeles, CA 90023, where the friendly staff went above and beyond to make my visit truly enjoyable!', 'gmap_0')
('Luis P.', '2017-07-10 22:12:19', 5, 'I had an amazing experience at this charming café in Los Angeles, where the friendly staff and delicious pastries made my day truly special!', 'gmap_0')
('His Mama Cakez', 'May 19, 2021 at 03:5

In [9]:
import pandas as pd
import sqlite3

# 将 la_data 转为 DataFrame
la_df = pd.DataFrame(la_data)

# 提取所有 gmap_id
gmap_ids = la_df["gmap_id"].tolist()

# 连接 review_query.db 数据库
conn = sqlite3.connect("../query_dataset/review_query.db")

# 查询匹配 gmap_id 的所有评分记录
placeholder = ','.join('?' for _ in gmap_ids)
query = f"""
    SELECT gmap_id, rating
    FROM review
    WHERE gmap_id IN ({placeholder})
"""
review_df = pd.read_sql_query(query, conn, params=gmap_ids)

# 计算平均评分
avg_rating = review_df.groupby("gmap_id")["rating"].mean().reset_index()
avg_rating.rename(columns={"rating": "avg_rating"}, inplace=True)

# 合并回原始洛杉矶商家数据
la_with_rating = la_df.merge(avg_rating, on="gmap_id", how="left")

# 按平均评分降序排列，取前5名
top5 = la_with_rating.sort_values(by="avg_rating", ascending=False).head(5)

print(top5)


                name  gmap_id  \
6  Widows Peak Salon  gmap_77   
0       City Textile   gmap_0   
3   Nobel Textile Co   gmap_3   
1       San Soo Dang   gmap_1   
2       Nova Fabrics   gmap_2   

                                         description  num_of_reviews  \
6   This modern hair salon in Los Angeles, CA 900...              35   
0   This textile exporter located in Los Angeles,...               6   
3   This fabric store in Los Angeles, CA 90021 of...               7   
1   This vibrant Korean restaurant in Los Angeles...              18   
2   This fabric store in Los Angeles, CA 90021 of...               6   

                                               hours  \
6  [["Thursday", "11AM–8PM"], ["Friday", "11AM–7P...   
0                                               null   
3  [["Thursday", "9AM–5PM"], ["Friday", "9AM–5PM"...   
1  [["Thursday", "6:30AM–6PM"], ["Friday", "6:30A...   
2  [["Thursday", "9AM–5PM"], ["Friday", "9AM–5PM"...   

                               